In [4]:
#!/usr/bin/env python
"""
Fluxo completo:

1) Seleciona a melhor cena ASTER L1T do Planetary Computer para uma folha
   cartográfica (carto.folhas_cartograficas), segundo:

   - coleção: aster-l1t
   - AOI: geom da folha em WGS84 (EPSG:4326)
   - fração de cobertura da folha ≥ aster_min_coverage (ex.: 0.9 = 90 %)
   - opcional: eo:cloud_cover < aster_max_cloud (se não for None)
   - ranking: |data_item − data_alvo| (dias), depois cloud_cover, depois cobertura

2) Usa a data da melhor cena ASTER como data de referência e seleciona a melhor
   cena Sentinel-2 L2A para a mesma folha, segundo:

   - coleção: sentinel-2-l2a
   - AOI: mesma geom da folha em WGS84
   - intervalo temporal de busca (search_datetime_s2), tipicamente a janela de
     operação do Sentinel-2 (ex.: "2015-06-23/2100-01-01")
   - fração de cobertura da folha ≥ s2_min_coverage (ex.: 0.9)
   - eo:cloud_cover < s2_max_cloud (ex.: 1.0 = 1 %)
   - ranking: |data_s2 − data_aster| (dias), depois cloud_cover, depois cobertura

Saída (no main):

- Lista de candidatos ASTER
- Melhor ASTER
- Lista de candidatos Sentinel-2
- Melhor Sentinel-2

Dependências (pip):

    pip install pystac-client planetary-computer shapely pyproj sqlalchemy psycopg2-binary pandas
"""

from __future__ import annotations

import json
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional, Tuple

import pandas as pd
from sqlalchemy import create_engine
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
from pyproj import Transformer
from pystac_client import Client
import planetary_computer as pc  # apenas para manter consistência de ambiente (não usado para download aqui)


# ----------------------------------------------------------------------
# Configuração PostGIS
# ----------------------------------------------------------------------
PG_USER = "postgres"
PG_PASS = "postgres"
PG_HOST = "localhost"
PG_PORT = "5432"
PG_DB   = "geologia"

PG_CONN_STR = (
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)


# ----------------------------------------------------------------------
# Utilitários de data/hora
# ----------------------------------------------------------------------
def parse_target_date(date_str: str) -> datetime:
    """Converte 'YYYY-MM-DD' em datetime UTC (meia-noite)."""
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return dt.replace(tzinfo=timezone.utc)


def item_datetime(item) -> datetime:
    """Retorna datetime timezone-aware (UTC) de um pystac.Item."""
    if getattr(item, "datetime", None) is not None:
        dt = item.datetime
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt

    dt_str = item.properties["datetime"]
    if dt_str.endswith("Z"):
        dt_str = dt_str.replace("Z", "+00:00")
    dt = datetime.fromisoformat(dt_str)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt


# ----------------------------------------------------------------------
# Geometria da folha (PostGIS → GeoJSON WGS84)
# ----------------------------------------------------------------------
def get_folha_geom_geojson(
    codigo_folha: str,
    conn_str: str = PG_CONN_STR,
    target_srid: int = 4326,
) -> Dict[str, Any]:
    """
    Lê a geometria da folha em carto.folhas_cartograficas e devolve GeoJSON
    em EPSG:4326.

    Supõe:
      - coluna 'codigo' (varchar)
      - coluna 'geom' (geometry)
    """
    engine = create_engine(conn_str)

    sql = f"""
    SELECT
        ST_AsGeoJSON(ST_Transform(geom, {target_srid})) AS geom_geojson
    FROM carto.folhas_cartograficas
    WHERE codigo = %(codigo)s
    LIMIT 1;
    """

    df = pd.read_sql(sql, engine, params={"codigo": codigo_folha})

    if df.empty or df.loc[0, "geom_geojson"] is None:
        raise RuntimeError(
            f"Não foi encontrada geometria para a folha '{codigo_folha}' "
            "em carto.folhas_cartograficas."
        )

    return json.loads(df.loc[0, "geom_geojson"])


def build_local_equal_area_transformer(
    geom_geojson: Dict[str, Any]
) -> Transformer:
    """
    Projeção LAEA (equal-area) centrada no centróide da geometria (WGS84).
    """
    geom = shape(geom_geojson)
    lon0, lat0 = geom.centroid.x, geom.centroid.y

    laea_proj = (
        f"+proj=laea +lat_0={lat0} +lon_0={lon0} "
        "+datum=WGS84 +units=m +no_defs"
    )
    transformer = Transformer.from_crs(
        "EPSG:4326",
        laea_proj,
        always_xy=True,
    )
    return transformer


def coverage_fraction(
    folha_geom_geojson: Dict[str, Any],
    item_geom_geojson: Dict[str, Any],
    transformer: Transformer,
) -> float:
    """
    Fração de cobertura da folha pela footprint do item:

        frac = área( folha ∩ item ) / área( folha )

    calculada em projeção equal-area local.
    """
    folha_geom = shape(folha_geom_geojson)
    item_geom = shape(item_geom_geojson)

    project = transformer.transform
    folha_proj = shp_transform(project, folha_geom)
    item_proj = shp_transform(project, item_geom)

    inter = folha_proj.intersection(item_proj)

    if folha_proj.is_empty or inter.is_empty:
        return 0.0

    return inter.area / folha_proj.area


# ----------------------------------------------------------------------
# Cloud cover via metadado STAC
# ----------------------------------------------------------------------
def get_cloud_cover(item) -> Optional[float]:
    """
    Retorna porcentagem de nuvem de um item STAC.

    Prioridade:
      - properties['eo:cloud_cover']
      - properties['s2:cloud_cover']
    """
    props = item.properties or {}

    for key in ("eo:cloud_cover", "s2:cloud_cover"):
        if key in props and props[key] is not None:
            try:
                return float(props[key])
            except Exception:
                continue
    return None


# ----------------------------------------------------------------------
# ASTER: busca melhor cena para a folha
# ----------------------------------------------------------------------
def search_aster_for_folha(
    codigo_folha: str,
    target_date_str: str,
    collection_id: str = "aster-l1t",
    stac_url: str = "https://planetarycomputer.microsoft.com/api/stac/v1",
    search_datetime: str = "2000-01-01/2015-12-31",
    min_coverage: float = 0.90,
    max_cloud: Optional[float] = None,
    max_items: int = 2000,
) -> Tuple[List[Dict[str, Any]], Optional[Dict[str, Any]]]:
    """
    Seleciona cenas ASTER L1T para uma folha cartográfica.

    Critérios:
      - intersects = geom da folha (WGS84)
      - datetime ∈ search_datetime
      - cobertura_folha ≥ min_coverage
      - se max_cloud não for None: cloud_cover < max_cloud
      - ranking: (|data_item − data_alvo| (dias), cloud_cover, -coverage)

    Retorno:
      (candidates, best)
        candidates: lista de dicts com campos:
            - id
            - datetime
            - delta_days
            - coverage_fraction
            - coverage_percent
            - cloud_cover
        best: dict do melhor item ou None
    """
    folha_geom_geojson = get_folha_geom_geojson(codigo_folha)
    folha_shape = shape(folha_geom_geojson)
    print(
        f"[ASTER] Folha '{codigo_folha}' carregada. "
        f"Área WGS84 (aprox): {folha_shape.area:.6f} (graus²)"
    )

    transformer = build_local_equal_area_transformer(folha_geom_geojson)

    cat = Client.open(stac_url)

    print(
        f"[ASTER] Buscando itens na coleção '{collection_id}' que intersectem a folha..."
    )
    search = cat.search(
        collections=[collection_id],
        intersects=folha_geom_geojson,
        datetime=search_datetime,
        max_items=max_items,
    )
    items = list(search.get_items())
    print(
        f"[ASTER] Total de itens retornados pelo STAC (antes de filtros locais): {len(items)}"
    )

    if not items:
        return [], None

    target_date = parse_target_date(target_date_str).date()
    candidates: List[Dict[str, Any]] = []

    for item in items:
        dt = item_datetime(item)
        dt_date = dt.date()
        delta_days = abs((dt_date - target_date).days)

        cloud = get_cloud_cover(item)
        if cloud is None:
            continue
        if (max_cloud is not None) and (cloud >= max_cloud):
            continue

        frac = coverage_fraction(
            folha_geom_geojson=folha_geom_geojson,
            item_geom_geojson=item.geometry,
            transformer=transformer,
        )
        if frac < min_coverage:
            continue

        candidates.append(
            {
                "id": item.id,
                "datetime": dt.isoformat(),
                "delta_days": delta_days,
                "coverage_fraction": frac,
                "coverage_percent": frac * 100.0,
                "cloud_cover": cloud,
                "item": item,
            }
        )

    if not candidates:
        print(
            "[ASTER] Nenhum item atendeu simultaneamente a:\n"
            f"  coverage >= {min_coverage*100:.1f}%"
            + (f" e cloud < {max_cloud:.2f}%." if max_cloud is not None else ".")
        )
        return [], None

    candidates.sort(
        key=lambda d: (d["delta_days"], d["cloud_cover"], -d["coverage_fraction"])
    )

    print("\n[ASTER] Itens candidatos (ordenados):")
    for c in candidates:
        print(
            f"  id={c['id']}, "
            f"datetime={c['datetime']}, "
            f"delta_days={c['delta_days']}, "
            f"coverage={c['coverage_percent']:.2f}%, "
            f"cloud={c['cloud_cover']:.2f}%"
        )

    best = candidates[0]
    print(
        "\n[ASTER] Melhor item (mais próximo da data alvo, com filtros aplicados):\n"
        f"  id={best['id']}\n"
        f"  datetime={best['datetime']}\n"
        f"  delta_days={best['delta_days']}\n"
        f"  coverage={best['coverage_percent']:.2f}%\n"
        f"  cloud={best['cloud_cover']:.2f}%"
    )

    return candidates, best


# ----------------------------------------------------------------------
# Sentinel-2: busca melhor cena para a folha, referenciada na data ASTER
# ----------------------------------------------------------------------
def search_sentinel2_for_folha(
    codigo_folha: str,
    reference_datetime: datetime,
    collection_id: str = "sentinel-2-l2a",
    stac_url: str = "https://planetarycomputer.microsoft.com/api/stac/v1",
    search_datetime: str = "2015-06-23/2100-01-01",
    min_coverage: float = 0.90,
    max_cloud: float = 1.0,
    max_items: int = 2000,
) -> Tuple[List[Dict[str, Any]], Optional[Dict[str, Any]]]:
    """
    Seleciona cenas Sentinel-2 L2A para a mesma folha, usando como referência
    temporal a data/hora da cena ASTER escolhida.

    Critérios:
      - intersects = geom da folha (WGS84)
      - datetime ∈ search_datetime (janela de operação do Sentinel-2)
      - cobertura_folha ≥ min_coverage
      - cloud_cover < max_cloud
      - ranking: (|data_s2 − data_ref_aster| (dias), cloud_cover, -coverage)

    Retorno:
      (candidates, best) no mesmo formato da função ASTER.
    """
    folha_geom_geojson = get_folha_geom_geojson(codigo_folha)
    folha_shape = shape(folha_geom_geojson)
    print(
        f"[S2] Folha '{codigo_folha}' carregada. "
        f"Área WGS84 (aprox): {folha_shape.area:.6f} (graus²)"
    )

    transformer = build_local_equal_area_transformer(folha_geom_geojson)
    cat = Client.open(stac_url)

    print(
        f"[S2] Buscando itens na coleção '{collection_id}' que intersectem a folha..."
    )
    search = cat.search(
        collections=[collection_id],
        intersects=folha_geom_geojson,
        datetime=search_datetime,
        max_items=max_items,
    )
    items = list(search.get_items())
    print(
        f"[S2] Total de itens retornados pelo STAC (antes de filtros locais): {len(items)}"
    )

    if not items:
        return [], None

    ref_date = reference_datetime.date()
    candidates: List[Dict[str, Any]] = []

    for item in items:
        dt = item_datetime(item)
        dt_date = dt.date()
        delta_days = abs((dt_date - ref_date).days)

        cloud = get_cloud_cover(item)
        if cloud is None:
            continue
        if cloud >= max_cloud:
            continue

        frac = coverage_fraction(
            folha_geom_geojson=folha_geom_geojson,
            item_geom_geojson=item.geometry,
            transformer=transformer,
        )
        if frac < min_coverage:
            continue

        candidates.append(
            {
                "id": item.id,
                "datetime": dt.isoformat(),
                "delta_days": delta_days,
                "coverage_fraction": frac,
                "coverage_percent": frac * 100.0,
                "cloud_cover": cloud,
                "item": item,
            }
        )

    if not candidates:
        print(
            "[S2] Nenhum item atendeu simultaneamente a:\n"
            f"  coverage >= {min_coverage*100:.1f}% e cloud < {max_cloud:.2f}%."
        )
        return [], None

    candidates.sort(
        key=lambda d: (d["delta_days"], d["cloud_cover"], -d["coverage_fraction"])
    )

    print("\n[S2] Itens candidatos (ordenados):")
    for c in candidates:
        print(
            f"  id={c['id']}, "
            f"datetime={c['datetime']}, "
            f"delta_days={c['delta_days']}, "
            f"coverage={c['coverage_percent']:.2f}%, "
            f"cloud={c['cloud_cover']:.2f}%"
        )

    best = candidates[0]
    print(
        "\n[S2] Melhor item (mais próxima da data ASTER, com filtros aplicados):\n"
        f"  id={best['id']}\n"
        f"  datetime={best['datetime']}\n"
        f"  delta_days={best['delta_days']}\n"
        f"  coverage={best['coverage_percent']:.2f}%\n"
        f"  cloud={best['cloud_cover']:.2f}%"
    )

    return candidates, best


# ----------------------------------------------------------------------
# Pipeline integrado: ASTER -> Sentinel-2
# ----------------------------------------------------------------------
def run_aster_and_s2_for_folha(
    codigo_folha: str,
    aster_target_date_str: str = "2008-07-15",
    aster_search_datetime: str = "2000-01-01/2015-12-31",
    aster_min_coverage: float = 0.90,
    aster_max_cloud: Optional[float] = None,  # por ex. 20.0 se quiser limitar
    s2_search_datetime: str = "2015-06-23/2100-01-01",
    s2_min_coverage: float = 0.90,
    s2_max_cloud: float = 1.0,
) -> Dict[str, Any]:
    """
    Executa o fluxo completo:

    1) Busca melhor cena ASTER para a folha e data alvo.
    2) Usa a data/hora dessa cena ASTER como referência para buscar a melhor
       Sentinel-2 L2A com < s2_max_cloud % de nuvem e cobertura ≥ s2_min_coverage.

    Retorno:
      dicionário com:
        - 'aster_candidates'
        - 'aster_best'
        - 's2_candidates'
        - 's2_best'
    """
    aster_candidates, aster_best = search_aster_for_folha(
        codigo_folha=codigo_folha,
        target_date_str=aster_target_date_str,
        collection_id="aster-l1t",
        stac_url="https://planetarycomputer.microsoft.com/api/stac/v1",
        search_datetime=aster_search_datetime,
        min_coverage=aster_min_coverage,
        max_cloud=aster_max_cloud,
        max_items=2000,
    )

    if not aster_best:
        print("\n[NENHUMA ASTER ADEQUADA] Abortando busca Sentinel-2.")
        return {
            "aster_candidates": aster_candidates,
            "aster_best": None,
            "s2_candidates": [],
            "s2_best": None,
        }

    aster_best_item = aster_best["item"]
    aster_best_dt = item_datetime(aster_best_item)
    print(
        f"\n[PIPELINE] Melhor ASTER selecionada em "
        f"{aster_best_dt.isoformat()} -> referência para Sentinel-2."
    )

    s2_candidates, s2_best = search_sentinel2_for_folha(
        codigo_folha=codigo_folha,
        reference_datetime=aster_best_dt,
        collection_id="sentinel-2-l2a",
        stac_url="https://planetarycomputer.microsoft.com/api/stac/v1",
        search_datetime=s2_search_datetime,
        min_coverage=s2_min_coverage,
        max_cloud=s2_max_cloud,
        max_items=2000,
    )

    return {
        "aster_candidates": aster_candidates,
        "aster_best": aster_best,
        "s2_candidates": s2_candidates,
        "s2_best": s2_best,
    }


# ----------------------------------------------------------------------
# Exemplo de uso direto
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Folha cartográfica (quadricula)
    codigo_folha = "SB21_ZA_II1_NE"

    # Data alvo do levantamento aerogeofísico (centro de julho–agosto de 2008)
    aster_target_date_str = "2008-07-15"

    # Janela ampla para ASTER que inclui 2008
    aster_search_datetime = "2000-01-01/2015-12-31"

    # Janela de operação Sentinel-2
    s2_search_datetime = "2015-06-23/2100-01-01"

    resultado = run_aster_and_s2_for_folha(
        codigo_folha=codigo_folha,
        aster_target_date_str=aster_target_date_str,
        aster_search_datetime=aster_search_datetime,
        aster_min_coverage=0.90,
        aster_max_cloud=5,    # ou por exemplo 20.0 para restringir nuvem
        s2_search_datetime=s2_search_datetime,
        s2_min_coverage=0.90,
        s2_max_cloud=1.0,        # < 1% de nuvem na folha
    )

    # DataFrames de candidatos (se quiser inspecionar no notebook)
    if resultado["aster_candidates"]:
        df_aster = pd.DataFrame(
            [
                {k: v for k, v in c.items() if k != "item"}
                for c in resultado["aster_candidates"]
            ]
        )
        print("\n[ASTER] DataFrame de candidatos:")
        print(df_aster.head())

    if resultado["s2_candidates"]:
        df_s2 = pd.DataFrame(
            [
                {k: v for k, v in c.items() if k != "item"}
                for c in resultado["s2_candidates"]
            ]
        )
        print("\n[S2] DataFrame de candidatos:")
        print(df_s2.head())


[ASTER] Folha 'SB21_ZA_II1_NE' carregada. Área WGS84 (aprox): 0.015625 (graus²)
[ASTER] Buscando itens na coleção 'aster-l1t' que intersectem a folha...
[ASTER] Total de itens retornados pelo STAC (antes de filtros locais): 28

[ASTER] Itens candidatos (ordenados):
  id=AST_L1T_00307142001141443_20150418112215, datetime=2001-07-14T14:14:43.223000+00:00, delta_days=2558, coverage=96.55%, cloud=3.00%

[ASTER] Melhor item (mais próximo da data alvo, com filtros aplicados):
  id=AST_L1T_00307142001141443_20150418112215
  datetime=2001-07-14T14:14:43.223000+00:00
  delta_days=2558
  coverage=96.55%
  cloud=3.00%

[PIPELINE] Melhor ASTER selecionada em 2001-07-14T14:14:43.223000+00:00 -> referência para Sentinel-2.
[S2] Folha 'SB21_ZA_II1_NE' carregada. Área WGS84 (aprox): 0.015625 (graus²)
[S2] Buscando itens na coleção 'sentinel-2-l2a' que intersectem a folha...
[S2] Total de itens retornados pelo STAC (antes de filtros locais): 722

[S2] Itens candidatos (ordenados):
  id=S2A_MSIL2A_20170